In [6]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# =========================================================================
# PHASE 4: PORTABLE ECOSYSTEM ROUTING CONFIGURATIONS
# =========================================================================
PG_URL = "jdbc:postgresql://host.docker.internal:5433/ecommerce_db"
PG_PROPERTIES = {
    "user": "data_engineer",
    "password": "de_password123",
    "driver": "org.postgresql.Driver"
}

# 100% Portable endpoint using the defined network alias with flat Bolt protocol
NEO4J_URI = "bolt://neo4jhost:7687" # CHANGED FROM 'neo4j://' TO 'bolt://'
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "graph_password123"

# =========================================================================
# STEP 1: INITIALIZE SPARK SESSION WITH DELTA AND NEO4J DEPENDENCIES
# =========================================================================
SUBMIT_PACKAGES = (
    "io.delta:delta-spark_2.12:3.0.0,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "org.postgresql:postgresql:42.6.0,"
    "org.neo4j:neo4j-connector-apache-spark_2.12:5.3.1_for_spark_3"
)
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {SUBMIT_PACKAGES} pyspark-shell"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minio_password123"

spark = SparkSession.builder \
    .appName("Ecommerce_Graph_Topology_Sync_Engine") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://lh_minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print("🏁 Containerized PySpark Kernel safely awakened under verified standard 3.5.0.")

# =========================================================================
# STEP 2: STAGING OPERATIONAL DATA FRAMES
# =========================================================================
print("\n📥 Extracting user matrix from operational PostgreSQL instance via JDBC...")
pg_users_df = spark.read.jdbc(url=PG_URL, table="users", properties=PG_PROPERTIES)

print("📥 Extracting refined footprints from Delta Lake Silver tier via MinIO blocks...")
delta_silver_df = spark.read.format("delta").load("file:///home/jovyan/work/notebooks/data/silver/clickstream")

# =========================================================================
# STEP 3: EXECUTING IDEMPOTENT GRAPH WRITE PASSES (FORMAL TIER 5 STANDARDS)
# =========================================================================
print("\n🚀 Commencing topological graph sync operations inside Neo4j container...")

# Pass A: Write (:User) Nodes safely
pg_users_df.write \
    .format("org.neo4j.spark.DataSource") \
    .option("url", NEO4J_URI) \
    .option("authentication.type", "basic") \
    .option("authentication.basic.username", NEO4J_USER) \
    .option("authentication.basic.password", NEO4J_PASSWORD) \
    .option("labels", "User") \
    .option("node.properties", "user_id,first_name,email") \
    .option("node.keys", "user_id") \
    .mode("Append") \
    .save()
print("✅ Identifiable (:User) Node vectors securely mapped.")

# Pass B: Extract unique (:IPAddress) footprints from the telemetry layers
unique_ips_df = delta_silver_df.select("ip_address").distinct().withColumnRenamed("ip_address", "ip_string")

unique_ips_df.write \
    .format("org.neo4j.spark.DataSource") \
    .option("url", NEO4J_URI) \
    .option("authentication.type", "basic") \
    .option("authentication.basic.username", NEO4J_USER) \
    .option("authentication.basic.password", NEO4J_PASSWORD) \
    .option("labels", "IPAddress") \
    .option("node.properties", "ip_string") \
    .option("node.keys", "ip_string") \
    .mode("Append") \
    .save()
print("✅ Architectural (:IPAddress) Node networks established.")

# =========================================================================
# HARDENED PASS C: MULTI-THREADED RELATIONSHIP INJECTION VIA NATIVE CYPHER
# =========================================================================
print("\n🔗 Executing query-driven relationship injections...")

# Explicitly leveraging native Cypher MERGE template to guarantee edge creation
delta_silver_df.write \
    .format("org.neo4j.spark.DataSource") \
    .option("url", NEO4J_URI) \
    .option("authentication.type", "basic") \
    .option("authentication.basic.username", NEO4J_USER) \
    .option("authentication.basic.password", NEO4J_PASSWORD) \
    .option("query", """
        MATCH (u:User {user_id: event.user_id})
        MATCH (ip:IPAddress {ip_string: event.ip_address})
        MERGE (u)-[r:CLICKED_FROM]->(ip)
        SET r.device_type = event.device_type,
            r.ingested_at = event.ingested_at
    """) \
    .mode("Append") \
    .save()

print("✅ Directional [:CLICKED_FROM] edges successfully locked into disk blocks.")
print("\n🎉 PoC State Sync Completed Successfully!")



🏁 Containerized PySpark Kernel safely awakened under verified standard 3.5.0.

📥 Extracting user matrix from operational PostgreSQL instance via JDBC...
📥 Extracting refined footprints from Delta Lake Silver tier via MinIO blocks...

🚀 Commencing topological graph sync operations inside Neo4j container...
✅ Identifiable (:User) Node vectors securely mapped.
✅ Architectural (:IPAddress) Node networks established.

🔗 Executing query-driven relationship injections...
✅ Directional [:CLICKED_FROM] edges successfully locked into disk blocks.

🎉 PoC State Sync Completed Successfully!
